In [0]:
bronze_schemapath  = "`bronze_catalog`.default"
silver_schemapath = "`silver_catalog`.default"

In [0]:
tables = spark.catalog.listTables(dbName=bronze_schemapath)

In [0]:
fivetran_ingested_column_to_remove = [
    "_file", 	
    "_fivetran_synced", 
    "_modified",
    "_line"
]

In [0]:
def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.product_table")
df_clean = remove_fivetran_ingested_column(df)
#df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.product_table")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, to_date, col, split, expr

df = spark.read.table(f"{silver_schemapath}.product_table")
df = df.withColumn("created_date", to_date(col("created_date"), "dd-MM-yyyy"))
df = df.withColumn(
    "product_internal_id",
    split(col("product_name"), " ")[2]
)
df = df.dropDuplicates()
display(df)
#df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.product_table")